<h2>Project: Smart Parking Prediction System</h2>
<h3>Dataset: Melbourne On-Street Parking Bay Sensors</h3>

<h2>1.Import Required Libraries</h2>

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif


<h2>2.Data preproceesing</h2>

In [2]:
# Load Parking Data 
df = pd.read_csv('on-street-parking-bay-sensors.csv')

# Timestamp Processing
df['Status_Timestamp'] = pd.to_datetime(df['Status_Timestamp'], errors='coerce')
df.dropna(subset=['Status_Timestamp'], inplace=True)

# Time-Based Features 
df['Hour'] = df['Status_Timestamp'].dt.hour
df['DayOfWeek'] = df['Status_Timestamp'].dt.dayofweek
df['Is_Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

# Target Encoding 
df['Occupancy'] = df['Status_Description'].map({'Present': 1, 'Unoccupied': 0})
df.dropna(subset=['Occupancy'], inplace=True)

# Optional Time Flags 
df['Is_Morning'] = df['Hour'].between(6, 11).astype(int)
df['Is_Afternoon'] = df['Hour'].between(12, 17).astype(int)
df['Is_Evening'] = df['Hour'].between(18, 22).astype(int)
df['Is_LateNight'] = df['Hour'].between(23, 5).astype(int)

# Remove Constant Features
constant_columns = [col for col in df.columns if df[col].nunique() == 1]
print("Constant Features:", constant_columns)
df.drop(columns=constant_columns, inplace=True)

#Feature Selection
features = ['Hour', 'DayOfWeek', 'Is_Weekend', 'Is_Morning', 'Is_Afternoon', 'Is_Evening']
X = df[features]
y = df['Occupancy']

selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X, y)
scores = selector.scores_

feature_scores = pd.DataFrame({'Feature': features, 'F_Score': scores})
print(feature_scores.sort_values(by='F_Score', ascending=False))

#Keep Only Important Features
important_features = feature_scores[feature_scores['F_Score'] > 1]['Feature'].tolist()
X_final = df[important_features]
print(f"Selected Important Features: {important_features}")





#Save Preprocessed Data
preprocessed_df = df[['Status_Timestamp'] + important_features + ['Occupancy']]
preprocessed_df.to_csv('Parking_Preprocessed.csv', index=False)
print(f"Preprocessed data saved to Parking_Preprocessed.csv, shape: {preprocessed_df.shape}")

#Ready for Modeling
print("Data ready for modeling")

Constant Features: ['Is_LateNight']
        Feature     F_Score
3    Is_Morning  234.588684
5    Is_Evening  101.694786
0          Hour   35.743614
4  Is_Afternoon   31.306875
1     DayOfWeek   25.146412
2    Is_Weekend    0.128139
Selected Important Features: ['Hour', 'DayOfWeek', 'Is_Morning', 'Is_Afternoon', 'Is_Evening']
Preprocessed data saved to Parking_Preprocessed.csv, shape: (3309, 7)
Data ready for modeling


<h2>3.Dataset Splitting, Scaling, and Saving</h2>

In [3]:
from sklearn.model_selection import train_test_split
import pandas as pd

#Load Preprocessed Data
df = pd.read_csv('Parking_Preprocessed.csv')
print(f"Loaded data shape: {df.shape}")

#Features & Target
features = [col for col in df.columns if col not in ['Status_Timestamp', 'Occupancy']]
X = df[features]
y = df['Occupancy']

#Split: Train+Val & Test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#Split: Train & Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}, Test shape: {X_test.shape}")

#Save All Splits
X_train.to_csv("X_train.csv", index=False)
X_val.to_csv("X_val.csv", index=False)
X_test.to_csv("X_test.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("All datasets saved: Train, Validation, Test.")


Loaded data shape: (3309, 7)
Train shape: (2117, 5), Validation shape: (530, 5), Test shape: (662, 5)
All datasets saved: Train, Validation, Test.


In [4]:
print("Train label distribution：")
print(y_train.value_counts(normalize=True))

print("\nValidation label distribution：")
print(y_val.value_counts(normalize=True))

print("\nTest label distribution：")
print(y_test.value_counts(normalize=True))


Train label distribution：
Occupancy
0    0.691072
1    0.308928
Name: proportion, dtype: float64

Validation label distribution：
Occupancy
0    0.690566
1    0.309434
Name: proportion, dtype: float64

Test label distribution：
Occupancy
0    0.690332
1    0.309668
Name: proportion, dtype: float64
